In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
from scipy.spatial import distance_matrix
from itertools import permutations

# Nearest Neighbor Algorithm
def nearest_neighbor(distance_matrix):
    n = distance_matrix.shape[0]
    visited = [False] * n
    path = []
    current_node = 0
    total_cost = 0

    for _ in range(n - 1):
        path.append(current_node)
        visited[current_node] = True
        min_distance = np.inf
        next_node = -1

        for i in range(n):
            if not visited[i] and distance_matrix[current_node][i] < min_distance:
                min_distance = distance_matrix[current_node][i]
                next_node = i

        total_cost += min_distance
        current_node = next_node

    path.append(current_node)
    total_cost += distance_matrix[current_node][0]

    return path, total_cost

In [ ]:
import numpy as np
from scipy.spatial import distance_matrix
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

def solve_tsp_and_get_route_cost(dist_matrix):
    """
    Solves the TSP for n randomly generated points in a 2D space,
    and returns the order of nodes and the scaled-back cost of travel.
    
    :param n: Number of points (nodes) to generate.
    :return: Tuple (route as np.ndarray, total cost of travel)
    """
    # Create the data model for OR-Tools
    data = {}
    data['distance_matrix'] = dist_matrix
    data['num_vehicles'] = 1
    data['depot'] = 0

    # Setup and solve the TSP
    manager = pywrapcp.RoutingIndexManager(len(data['distance_matrix']), data['num_vehicles'], data['depot'])
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        """Returns the distance between the two nodes."""
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return data['distance_matrix'][from_node][to_node]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)
    search_parameters.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH)
    time_limit = 5# int(dist_matrix.shape[0] * 0.01) if dist_matrix.shape[0] > 1 else 1
    search_parameters.time_limit.seconds = time_limit
    solution = routing.SolveWithParameters(search_parameters)

    if solution:
        # Retrieve the route and scale back the total cost
        index = routing.Start(0)
        route = [manager.IndexToNode(index)]
        total_cost = 0
        while not routing.IsEnd(index):
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route.append(manager.IndexToNode(index))
            total_cost += routing.GetArcCostForVehicle(previous_index, index, 0)
        total_cost /= 100  # Scale back by dividing by 100
        return np.array(route[:-1]), total_cost/100  # Exclude the return to the depot
    else:
        return None, None

In [ ]:
import fast_tsp
def fastTSP(dist_matrix, precision=0.01):
    """
    Solves the TSP for n randomly generated points in a 2D space,
    and returns the order of nodes and the scaled-back cost of travel.
    
    :param n: Number of points (nodes) to generate.
    :return: Tuple (route as np.ndarray, total cost of travel)
    """
    # Create the data model for OR-Tools
    dist_matrix /= precision  # Scale the distances to integers
    # into integers
    dist_matrix = dist_matrix.astype(int)
    tour = fast_tsp.find_tour(dist_matrix)
    return tour, fast_tsp.compute_cost(tour, dist_matrix) * precision

In [ ]:
#tsp solver using linear_sum_assignment
def tsp_solver(dist_matrix):
    from scipy.optimize import linear_sum_assignment
    row_ind, col_ind = linear_sum_assignment(dist_matrix)
    return col_ind, dist_matrix[col_ind, col_ind].sum()

In [ ]:
import time
import numpy as np
from scipy.spatial import distance_matrix

# Generate n random points
n = 1000
points = np.random.rand(n, 2)

# Compute the distance matrix
dist_matrix = distance_matrix(points, points)

# Ensure positive weights and satisfy the triangle inequality
dist_matrix = np.maximum(dist_matrix, dist_matrix.T)
dist_matrix = np.maximum(dist_matrix, np.zeros_like(dist_matrix))

# Solve using Nearest Neighbor Algorithm
print("Nearest Neighbor Algorithm:")
start_time = time.time()
nn_path, nn_cost = nearest_neighbor(dist_matrix)
print("--- %s seconds ---" % (time.time() - start_time))
print("Path:", nn_path)
print("Cost:", nn_cost)

# Solve using Fast TSP
print("Fast TSP Solver:")
start_time = time.time()
fast_path, fast_cost = fastTSP(dist_matrix, precision=0.0001)
print("--- %s seconds ---" % (time.time() - start_time))
print("Path:", fast_path)
print("Cost:", fast_cost)

In [ ]:
import numpy as np
from scipy.spatial import distance_matrix

n_list = [100*(i+1) for i in range(50)]

nn_times = []
opt_times = []
nn_costs = []
opt_costs = []

for n in n_list:
    points = np.random.rand(n, 2)
    dist_matrix = distance_matrix(points, points)

    start_time = time.time()
    nn_path, nn_cost = fastTSP(dist_matrix, precision=0.0001)
    nn_times.append(time.time() - start_time)
    nn_costs.append(nn_cost)

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(n_list, nn_times, label="Nearest Neighbor")
plt.xlabel("Number of Points")
plt.ylabel("Time (s)")
plt.title("Execution Time vs. Number of Points")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(n_list, nn_costs, label="Nearest Neighbor")
plt.xlabel("Number of Points")
plt.ylabel("Cost")
plt.title("Cost vs. Number of Points")
plt.legend()
plt.show()
